In [21]:
import json
import unicodedata
from pathlib import Path
from difflib import SequenceMatcher
import re
from typing import List, Tuple
import os
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Any
from typing import List, Tuple
from collections import defaultdict
# Regex for ATT&CK technique IDs e.g. T1566 or T1566.001
ATTACK_ID_RE = re.compile(
    r"\b(?:T\d{4}(?:\.\d{3})?|TA\d{4})\b",
    flags=re.IGNORECASE
)
SENTENCE_SPLIT_RE = re.compile(r'(?<=[\.\?\!])\s+')

def normalize_text(s: str) -> str:
    if s is None:
        return ""
    try:
        s = unicodedata.normalize("NFKC", s)
    except Exception:
        pass
    return s.lower()


def find_id_sentences(text: str, valid_ids: set = None):
    """Return sentences containing ATT&CK IDs (validated)."""
    sentences = split_sentences(text)
    hits = []
    for sent in sentences:
        for m in ATTACK_ID_RE.finditer(sent):
            tid = m.group(0).upper()
            if valid_ids and tid not in valid_ids:
                continue
            hits.append({
                "id": tid,
                "sentence": sent.strip()
            })
    return hits


def find_name_sentences(text_norm: str, id_to_name: dict):
    """Return sentences containing ATT&CK technique names."""
    sentences = split_sentences(text_norm)
    hits = []
    for tid, name in id_to_name.items():
        name_norm = normalize_text(name)
        for sent in sentences:
            if name_norm in sent:
                hits.append({
                    "id": tid,
                    "name": name,
                    "sentence": sent.strip()
                })
    return hits
def _merge_fragmented_lines(lines: List[str], max_buffer_words: int = 200) -> List[str]:
    """
    Merge short/fragmented lines into larger lines by collecting successive non-empty lines
    until we encounter a line that ends with sentence punctuation or an explicit blank line.
    This helps with text which has lots of single-word lines.
    """
    merged = []
    buf = []
    for raw in lines:
        line = (raw or "").strip()
        if not line:
            # flush on blank lines
            if buf:
                merged.append(" ".join(buf).strip())
                buf = []
            continue

        buf.append(line)

        # heuristics to flush: if current line ends with punctuation or buffer is getting long
        if line.endswith((".", "?", "!", ":", ";")) or len(buf) >= max_buffer_words:
            merged.append(" ".join(buf).strip())
            buf = []

    if buf:
        merged.append(" ".join(buf).strip())
    return merged

def split_sentences(text: str) -> List[str]:
    """Return a list of sentences from text (keeps short fragments too)."""
    # Normalize newlines to spaces so sentences spanning lines are preserved
    if not text:
        return []
    text = text.replace("\r\n", " ").replace("\n", " ")
    # split and strip
    sents = [s.strip() for s in SENTENCE_SPLIT_RE.split(text) if s.strip()]
    return sents

def extract_ids_from_sentence(sentence: str, valid_ids: set = None) -> List[str]:
    """Return list of valid ATT&CK IDs found in a sentence."""
    ids = [m.group(0).upper() for m in ATTACK_ID_RE.finditer(sentence)]
    if valid_ids:
        ids = [tid for tid in ids if tid in valid_ids]
    return ids

def _norm_sent(s: str) -> str:
    # normalize for matching (lowercase + collapse whitespace)
    return " ".join(s.lower().split())
def sanitize_report_text(text: str,
                         min_block_len: int = 1,
                         window_size: int = 6,
                         window_threshold: int = 2,
                         line_id_threshold: int = 3
                         ) -> Tuple[str, List[List[str]]]:
    """
    Robust sanitizer:
      1) merges fragmented lines into coherent lines
      2) removes consecutive blocks of lines where each line contains an ATT&CK ID
      3) removes single lines/sentences containing too many ATT&CK IDs
      4) fallback: removes 'clusters' of lines where ID density in a sliding window >= threshold

    Returns (cleaned_text, removed_blocks)
    """
    if not text:
        return "", []

    # split original lines (preserve order)
    orig_lines = text.splitlines()

    # Step 1: Merge fragmented lines to better detect sentences/tables
    merged_lines = _merge_fragmented_lines(orig_lines)

    cleaned_lines = []
    buffer_block: List[str] = []
    removed_blocks: List[List[str]] = []

    def flush_block():
        nonlocal buffer_block
        if buffer_block:
            if len(buffer_block) < min_block_len:
                cleaned_lines.extend(buffer_block)
            else:
                removed_blocks.append(buffer_block[:])
            buffer_block = []

    # Step 2: Consecutive-line detection
    for line in merged_lines:
        if ATTACK_ID_RE.search(line):
            buffer_block.append(line)
        else:
            flush_block()
            cleaned_lines.append(line)
    flush_block()

    # Step 3: One-sentence / one-line with many IDs
    # Look for lines with >= line_id_threshold matches
    final_lines = []
    extra_removed = []
    for line in cleaned_lines:
        matches = ATTACK_ID_RE.findall(line)
        if len(matches) >= line_id_threshold:
            extra_removed.append(line)
        else:
            final_lines.append(line)

    if extra_removed:
        removed_blocks.append(extra_removed)
    cleaned_lines = final_lines

    # If we already removed something, return early
    if removed_blocks:
        return "\n".join(cleaned_lines), removed_blocks

    # Step 4: Fallback cluster detection — sliding window
    n = len(merged_lines)
    id_counts = [1 if ATTACK_ID_RE.search(l) else 0 for l in merged_lines]
    to_remove = [False] * n

    window_sum = sum(id_counts[:min(window_size, n)])
    if n:
        if window_sum >= window_threshold:
            for j in range(0, min(window_size, n)):
                to_remove[j] = True
    for i in range(1, n):
        prev_idx = i - 1
        remove_idx = i + window_size - 1
        window_sum = window_sum - id_counts[prev_idx]
        if remove_idx < n:
            window_sum += id_counts[remove_idx]
        if window_sum >= window_threshold:
            for j in range(i, min(i + window_size, n)):
                to_remove[j] = True

    cur_block = []
    final_cleaned = []
    for idx, line in enumerate(merged_lines):
        if to_remove[idx]:
            cur_block.append(line)
        else:
            if cur_block:
                removed_blocks.append(cur_block[:])
                cur_block = []
            final_cleaned.append(line)
    if cur_block:
        removed_blocks.append(cur_block[:])

    if not removed_blocks:
        return "\n".join(merged_lines), []

    return "\n".join(final_cleaned), removed_blocks

def build_ttp_sentences_from_id_hits(
    id_hits: List[Dict[str, Any]],
    sanitized_text: str,
    VALID_IDS:str,
    removed_blocks: List[List[str]] = None

) -> Dict[str, List[str]]:
    """
    Build mapping {technique_id: [sentences]} strictly from the sanitized text.
    - If id_hits are provided with 'sentence', keep only those whose sentence
      occurs in the sanitized text (so nothing from flushed blocks leaks in).
    - Otherwise, fallback to scanning the sanitized text by sentence.
    """
    ttp_sentences = defaultdict(list)

    # sentences from sanitized text (ground truth surface)
    sanitized_sents = [s.strip() for s in split_sentences(sanitized_text) if s.strip()]
    sanitized_sents_norm = {_norm_sent(s) for s in sanitized_sents}
    sanitized_lookup = { _norm_sent(s): s for s in sanitized_sents }  # norm -> original

    # path A: trust id_hits only if their sentence exists in sanitized text
    if id_hits and all(('id' in h and ('sentence' in h or 'sent' in h)) for h in id_hits):
        for h in id_hits:
            tid = h.get("id")
            sent = (h.get("sentence") or h.get("sent") or "").strip()
            if not tid or not sent:
                continue
            key = _norm_sent(sent)
            if key in sanitized_sents_norm:              # only keep if present post-sanitization
                ttp_sentences[tid].append(sanitized_lookup[key])
        # if we got anything valid, return it
        if ttp_sentences:
            return {k: v for k, v in ttp_sentences.items()}

    # path B: fallback — scan sanitized text sentences directly
    for s in sanitized_sents:
        for tid in extract_ids_from_sentence(s, VALID_IDS):
            ttp_sentences[tid].append(s)

    return {k: v for k, v in ttp_sentences.items()}


In [22]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

def main(csv_path: Path, out_dir: Path):
    # -----------------------
    # Load mapping JSON (tec_prefixed.json or tec.json)
    # -----------------------
    mapping_path = Path("/home/simonettos/thijs/data_augmentatio_stefano/rcatt/tec_prefixed.json")
    if not mapping_path.exists():
        mapping_path = Path("tec.json")

    if not mapping_path.exists():
        print(f"⚠️ Mapping file not found: tried tec_prefixed.json and tec.json")
        return

    try:
        id_to_name = json.loads(mapping_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"⚠️ Failed to load mapping file {mapping_path}: {e}")
        return

    VALID_IDS = set(id_to_name.keys())

    # -----------------------
    # Load CSV
    # -----------------------
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"⚠️ Could not read CSV {csv_path}: {e}")
        return

    if "Text" not in df.columns:
        print(f"⚠️ CSV does not contain a 'Text' column. Columns: {list(df.columns)}")
        return

    out_dir.mkdir(parents=True, exist_ok=True)

    # All columns except Text are “label columns” (as you described)
    label_cols = [c for c in df.columns if c != "Text"]

    processed = 0
    skipped = 0

    for i, row in df.iterrows():
        text = row.get("Text", "")

        if not isinstance(text, str) or not text.strip():
            skipped += 1
            continue

        try:
            raw = text

            # -----------------------
            # Extract row TTPs from columns (value=1/True)
            # -----------------------
            row_ttps_from_columns = []
            for c in label_cols:
                v = row[c]

                # Treat 1 / 1.0 / True / "1" as active
                is_active = False
                if isinstance(v, (int, np.integer)):
                    is_active = (v == 1)
                elif isinstance(v, (float, np.floating)):
                    is_active = (not np.isnan(v)) and (v == 1.0)
                elif isinstance(v, (bool, np.bool_)):
                    is_active = bool(v)
                elif isinstance(v, str):
                    is_active = (v.strip() == "1" or v.strip().lower() == "true")

                if is_active:
                    row_ttps_from_columns.append(c)

            # -----------------------
            # Sanitize
            # -----------------------
            sanitized, removed_blocks = sanitize_report_text(raw, min_block_len=1)
            raw_norm = normalize_text(sanitized)

            # -----------------------
            # Matchers (existing helpers)
            # -----------------------
            id_hits = find_id_sentences(raw, VALID_IDS)
            name_hits = find_name_sentences(raw, id_to_name)

            # -----------------------
            # Build JSON structure
            # -----------------------
            id_list_from_text = sorted({h.get("id") for h in id_hits if h.get("id")})
            name_list = sorted({h.get("name") for h in name_hits if h.get("name")})

            # Sentences for IDs found in text
            ttp_sentences = build_ttp_sentences_from_id_hits(id_hits, sanitized, VALID_IDS)

            # Union: IDs from text + IDs implied by active columns (only keep ones that look like Txxxx...)
            col_ids_normalized = []
            for t in row_ttps_from_columns:
                t = str(t).strip()
                # Keep both TA**** and T**** if your CSV contains tactics too
                if t.startswith("T") or t.startswith("TA"):
                    col_ids_normalized.append(t)

            id_list_union = sorted(set(id_list_from_text).union(col_ids_normalized))

            output_data = {
                "original_txt": raw,
                "sanitized_txt": sanitized,

                # extracted from text
                "ID_list": id_list_union,
                "Name_list": name_list,
                "TTP_sentences": ttp_sentences,
            }

            out_path = out_dir / f"row_{i:05d}.attack.json"
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)

            print(
                f"✅ Row {i}: wrote {out_path} "
                f"(text IDs: {len(id_list_from_text)}, col TTPs: {len(set(col_ids_normalized))}, union: {len(id_list_union)})"
            )
            processed += 1

        except Exception as e:
            print(f"⚠️ Error processing row {i}: {e}")
            skipped += 1

    print(f"\nFinished. Processed: {processed}. Skipped: {skipped}.")


if __name__ == "__main__":
    csv_path = Path("/home/simonettos/thijs/data_augmentatio_stefano/rcatt/training_data_original.csv")
    out_dir = Path("/home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out")
    main(csv_path.resolve(), out_dir.resolve())


✅ Row 0: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00000.attack.json (text IDs: 0, col TTPs: 26, union: 26)
✅ Row 1: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00001.attack.json (text IDs: 0, col TTPs: 2, union: 2)
✅ Row 2: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00002.attack.json (text IDs: 0, col TTPs: 20, union: 20)
✅ Row 3: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00003.attack.json (text IDs: 1, col TTPs: 2, union: 2)
✅ Row 4: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00004.attack.json (text IDs: 1, col TTPs: 2, union: 2)
✅ Row 5: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00005.attack.json (text IDs: 0, col TTPs: 2, union: 2)
✅ Row 6: wrote /home/simonettos/thijs/data_augmentatio_stefano/rcatt/attack_json_out/row_00006.attack.json (text IDs: 0, col TTPs: 2